In [ ]:
!pip install -q -U datasets soundfile pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 138.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


In [1]:
from google.colab import drive
from pathlib import Path
import os

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive already mounted.")

BASE_DIR = Path(
    "/content/drive/MyDrive/persian_asr_llm_error_propagation"
)

DATA_DIR = BASE_DIR / "data"

TSV_PATH = DATA_DIR / "test.tsv"
TAR_PATH = DATA_DIR / "test.tar.gz"

print("TSV exists:", TSV_PATH.exists())
print("TAR exists:", TAR_PATH.exists())

print("TSV:", TSV_PATH)
print("TAR:", TAR_PATH)

Google Drive already mounted.
TSV exists: True
TAR exists: True
TSV: /content/drive/MyDrive/persian_asr_llm_error_propagation/data/test.tsv
TAR: /content/drive/MyDrive/persian_asr_llm_error_propagation/data/test.tar.gz


In [2]:
import pandas as pd

COLUMNS = [
    "id",
    "file_name",
    "raw_transcription",
    "transcription",
    "character_transcription",
    "num_samples",
    "gender",
]

df = pd.read_csv(
    TSV_PATH,
    sep="\t",
    header=None,
    names=COLUMNS,
    dtype={
        "id": "int64",
        "file_name": "string",
        "raw_transcription": "string",
        "transcription": "string",
        "character_transcription": "string",
        "num_samples": "int64",
        "gender": "string",
    },
)

print("Shape:", df.shape)
display(df.head())

Shape: (871, 7)


,id,file_name,raw_transcription,transcription,character_transcription,num_samples,gender
0,1735,7913564082410055971.wav,محققان دانشگاه پرینستون آمریكا و دانشگاه اوپسا...,محققان دانشگاه پرینستون آمریکا و دانشگاه اوپسا...,م ح ق ق ا ن | د ا ن ش گ ا ه | پ ر ی ن س ت و ن ...,518400,MALE
1,1720,621633057978356932.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,193920,MALE
2,1720,13180061520623477685.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,205440,MALE
3,1720,16484235889057265269.wav,شکلات داغ در حد استانداردهای بلژیکی است. آبمیو...,شکلات داغ در حد استانداردهای بلژیکی است آبمیوه...,ش ک ل ا ت | د ا غ | د ر | ح د | ا س ت ا ن د ا ...,223680,MALE
4,1955,17985233135634044314.wav,MS نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,ms نوعی بیماری است که بر سیستم عصبی مرکزی تأثی...,m s | ن و ع ی | ب ی م ا ر ی | ا س ت | ک ه | ب ...,218880,MALE


In [3]:
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

print("\nMissing values:")
print(df.isna().sum())

print("\nGender:")
print(df["gender"].value_counts(dropna=False))

print("\nUnique WAV files:")
print(df["file_name"].nunique())

Rows: 871
Columns: ['id', 'file_name', 'raw_transcription', 'transcription', 'character_transcription', 'num_samples', 'gender']

Missing values:
id                         0
file_name                  0
raw_transcription          0
transcription              0
character_transcription    0
num_samples                0
gender                     0
dtype: int64

Gender:
gender
MALE    871
Name: count, dtype: Int64

Unique WAV files:
871


In [4]:
assert len(df.columns) == 7
assert df["file_name"].notna().all()
assert df["raw_transcription"].notna().all()
assert df["transcription"].notna().all()

print("TSV validation passed.")

TSV validation passed.


In [5]:
import tarfile

AUDIO_ROOT = DATA_DIR / "test_audio"

if not AUDIO_ROOT.exists():
    AUDIO_ROOT.mkdir(parents=True)

    with tarfile.open(TAR_PATH, "r:gz") as tar:
        tar.extractall(AUDIO_ROOT)

    print("Audio extracted.")
else:
    print("Audio directory already exists.")

print(AUDIO_ROOT)

/tmp/ipykernel_1639/3276319558.py:9: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(AUDIO_ROOT)


Audio extracted.
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio


In [6]:
for path in sorted(AUDIO_ROOT.iterdir()):
    print(path)

/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test


In [7]:
wav_files = list(
    AUDIO_ROOT.rglob("*.wav")
)

print("WAV files found:", len(wav_files))

for p in wav_files[:10]:
    print(p)

WAV files found: 871
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/10039392309700583023.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/10051486205118679596.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/1010306930168345338.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/10124979281588232806.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/10129388121687242275.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/10130894543333251273.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/10140044754665750794.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/10197441927924455618.wav
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/10199698364975966614.wav
/content/drive/MyDrive/persian_asr_llm_error_propagat

In [8]:
wav_map = {
    p.name: str(p)
    for p in wav_files
}

df["audio_path"] = (
    df["file_name"]
    .map(wav_map)
)

print(
    "Matched audio:",
    df["audio_path"].notna().sum(),
    "/",
    len(df)
)

Matched audio: 871 / 871


In [9]:
missing_audio = df[
    df["audio_path"].isna()
]

print(
    "Missing audio files:",
    len(missing_audio)
)

if len(missing_audio):
    display(
        missing_audio[
            ["id", "file_name"]
        ].head(20)
    )

Missing audio files: 0


In [10]:
import soundfile as sf

sample_path = df.iloc[0]["audio_path"]

info = sf.info(sample_path)

print(info)

/content/drive/MyDrive/persian_asr_llm_error_propagation/data/test_audio/test/7913564082410055971.wav
samplerate: 16000 Hz
channels: 1
duration: 32.400 s
format: WAV (Microsoft) [WAV]
subtype: 32 bit float [FLOAT]


In [11]:
sample_rates = []
durations = []

for path in df["audio_path"]:
    info = sf.info(path)

    sample_rates.append(
        info.samplerate
    )

    durations.append(
        info.frames / info.samplerate
    )

df["sample_rate"] = sample_rates
df["duration_sec"] = durations

print("Sample rates:")
print(
    df["sample_rate"]
    .value_counts()
)

print("\nDuration statistics:")
print(
    df["duration_sec"]
    .describe()
)

Sample rates:
sample_rate
16000    871
Name: count, dtype: int64

Duration statistics:
count    871.000000
mean      15.294902
std        4.681832
min        5.340000
25%       11.940000
50%       14.520000
75%       17.640000
max       39.240000
Name: duration_sec, dtype: float64


In [12]:
print(
    "Total audio recordings:",
    len(df)
)

print(
    "Unique IDs:",
    df["id"].nunique()
)

print(
    "Unique raw transcripts:",
    df["raw_transcription"].nunique()
)

print(
    "Unique normalized transcripts:",
    df["transcription"].nunique()
)

Total audio recordings: 871
Unique IDs: 324
Unique raw transcripts: 324
Unique normalized transcripts: 324


In [14]:
recordings_per_id = (
    df.groupby("id")
    .size()
    .rename("num_recordings")
)

print(
    recordings_per_id
    .describe()
)

print("\nDistribution:")
print(
    recordings_per_id
    .value_counts()
    .sort_index()
)

count    324.000000
mean       2.688272
std        0.582286
min        1.000000
25%        2.750000
50%        3.000000
75%        3.000000
max        3.000000
Name: num_recordings, dtype: float64

Distribution:
num_recordings
1     20
2     61
3    243
Name: count, dtype: int64


In [15]:
texts_per_id = (
    df.groupby("id")[
        "transcription"
    ]
    .nunique()
)

print(
    "IDs with one transcript:",
    (texts_per_id == 1).sum()
)

print(
    "IDs with >1 transcript:",
    (texts_per_id > 1).sum()
)

IDs with one transcript: 324
IDs with >1 transcript: 0


Actual QA-generation dataset

In [16]:
unique_df = (
    df.sort_values(
        ["id", "file_name"]
    )
    .drop_duplicates(
        subset=["id"]
    )
    .copy()
)

unique_df["num_recordings"] = (
    unique_df["id"]
    .map(recordings_per_id)
)

print(
    "QA candidate texts:",
    len(unique_df)
)

QA candidate texts: 324


In [17]:
qa_candidates = unique_df[
    [
        "id",
        "raw_transcription",
        "transcription",
        "num_recordings",
    ]
].copy()

display(
    qa_candidates.head(10)
)

,id,raw_transcription,transcription,num_recordings
39,1660,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,2
731,1661,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,3
703,1662,آلیاژها اساساً‌ ترکیبی از دو یا چند فلز می‌باش...,آلیاژها اساساً ترکیبی از دو یا چند فلز می‌باشن...,3
425,1663,در Cochamó - مقصد برتر کوهنوردی در شیلی است، ک...,در cochamó - مقصد برتر کوهنوردی در شیلی است، ک...,1
154,1664,دو پودر خشک را مخلوط کنید و سپس با دستان خشک و...,دو پودر خشک را مخلوط کنید و سپس با دستان خشک و...,3
242,1665,این سند بر اساس این درز اطلاعاتی است و به اختل...,این سند بر اساس این درز اطلاعاتی است و به اختل...,3
869,1666,همچنین ممکن است بخواهید از توصیه‌های دولت‌های ...,همچنین ممکن است بخواهید از توصیه‌های دولت‌های ...,2
701,1667,بطور کلی، وقتی مدیران شروع به رهبری همکاران سا...,بطور کلی وقتی مدیران شروع به رهبری همکاران ساب...,3
75,1668,خرید کارت بازی که امکان ورود به منتخبی از پارک...,خرید کارت بازی که امکان ورود به منتخبی از پارک...,3
854,1670,به گفته «گلن کوشینگ» از ارزیابی جغرافیایی ایال...,به گفته «گلن کوشینگ» از ارزیابی جغرافیایی ایال...,1


In [19]:
qa_candidates["word_count"] = (
    qa_candidates[
        "transcription"
    ]
    .str.split()
    .str.len()
)

qa_candidates["char_count"] = (
    qa_candidates[
        "transcription"
    ]
    .str.len()
)

print(
    qa_candidates[
        [
            "word_count",
            "char_count",
            "num_recordings",
        ]
    ].describe(
        percentiles=[
            .10,
            .25,
            .50,
            .75,
            .90,
            .95,
        ]
    )
)

       word_count  char_count  num_recordings
count  324.000000       324.0      324.000000
mean    23.620370  121.271605        2.688272
std      8.840924   44.611293        0.582286
min      8.000000        41.0        1.000000
10%     14.000000        72.3        2.000000
25%     18.000000        90.0        2.750000
50%     22.000000       113.0        3.000000
75%     28.000000       146.0        3.000000
90%     35.000000       180.4        3.000000
95%     39.850000      198.85        3.000000
max     63.000000       317.0        3.000000


In [20]:
sample = qa_candidates.sample(
    n=min(
        25,
        len(qa_candidates)
    ),
    random_state=42,
)

for _, row in sample.iterrows():

    print("=" * 100)

    print(
        "ID:",
        row["id"]
    )

    print(
        "Recordings:",
        row["num_recordings"]
    )

    print(
        "Words:",
        row["word_count"]
    )

    print()

    print(
        row["raw_transcription"]
    )

    print()

ID: 1806
Recordings: 3
Words: 43

از آنجایی که آلودگی نوری در آغاز مشکلی محسوب نمی‌شد اما اکنون مشکل تلقی می‌شود، آنها معمولاً در شهرها و یا در محوطه دانشگاه ها قرار می گیرند و دسترسی به آنها نسبت به آنهایی که در دوران مدرن ساخته شده‌اند، آسانتر است.

ID: 1780
Recordings: 3
Words: 22

با استفاده از یوگا کوندالینی، انرژی کوندالینی (انرژی روشنگری) از طریق حرکات یوگا، تمرین های تنفسی، مانترا و تجسم بیدار می شود.

ID: 1812
Recordings: 3
Words: 32

با اینکه بنظر می‌رسد یک واکسن تجربی بتواند میزان مرگ و میر ایبولا را کاهش دهد، تا به امروز هیچ دارویی نبوده که به وضوح ثابت شود برای درمان عفونت کنونی است.

ID: 1670
Recordings: 1
Words: 47

به گفته «گلن کوشینگ» از ارزیابی جغرافیایی ایالات متحده (USGS) تیم نجوم دانشگاه آریزونای شمالی واقع در فلگستف آریزونا: «رفتار دمایی آنها به پایداری غارهای بزرگ روی کره زمین نیست که در آنها اغلب دمای نسبتاً ثابت حفظ می‌شود، ولی شبیه به چاله‌های عمیق در زمین می‌باشد،»

ID: 1859
Recordings: 3
Words: 34

در اواخر قرون وسطی، اروپای غربی شروع به توسعه مد کرد. یکی از

In [21]:
for _, row in (
    qa_candidates
    .sort_values("word_count")
    .head(15)
    .iterrows()
):

    print(
        row["id"],
        "|",
        row["word_count"],
        "|",
        row["raw_transcription"]
    )

1910 | 8 | پاریسی‌ها به خودپسندی، بی‌ادبی و تکبر شهره هستند.
1700 | 9 | زیر دریاوارها نازک‌تر و زیر زمین‌های مرتفع ضخیم‌تر است.
1755 | 9 | یک بمب در بیرون دفتر فرماندار کل منفجر شد.
1953 | 9 | او سپس به بیمارستان ادن بروک کمبریج منتقل شد.
1747 | 9 | کشتی گیران همکار نیز به لونا ادای احترام کردند.
1957 | 10 | او این شایعات را «حماقت و حرف مفت سیاسی» خواند.
1821 | 10 | ارتباط بین آسیب‌شناسی مغز و رفتار پشتوانه تحقیق دانشمندان است.
1941 | 10 | این زوجها می‌توانند طرح فرزندخواندگی را برای نوزادشان ترتیب دهند.
1871 | 10 | در آب و هوای گرم خاورمیانه، خانه زیاد اهمیت نداشت.
1679 | 11 | Aerosmith کنسرت‌های باقی مانده خود را در تور خود لغو کردند.
1778 | 11 | گزارش‌های تلویزیونی حاکی از وجود دود سفید بر فراز کارخانه است.
1926 | 12 | نظرات ارسطو در مورد همه مسائل علمی، از جمله روانشناسی پذیرفته شد.
1731 | 12 | شما در این مجمع‌الجزایر و دریاچه‌ها لزوماً به قایق تفریحی نیازی ندارید.
1772 | 12 | بنابراین، این احتمال هست که یادداشت‌ صرفاً به‌عنوان برچسب الصاق شده باشد.
1802 | 12 | اینترنت عناصر ارتبا

In [22]:
for _, row in (
    qa_candidates
    .sort_values(
        "word_count",
        ascending=False
    )
    .head(15)
    .iterrows()
):

    print(
        row["id"],
        "|",
        row["word_count"],
        "|",
        row["raw_transcription"]
    )

1937 | 63 | اما این نقشه‌ها تقریباً یک شبه از کار افتاد، وقتی که بیش از 800 هزار سرباز از ارتش سرخ اتحاد جماهیر شوروی وارد شدند و پس از هجوم به نواحی شرقی لهستان جبهه‌های بلاروسی و اوکراینی را ساختند، و بدین ترتیب عهدنامه صلح ریگا، پیمان عدم تجاوز بین شوروی و لهستان، و سایر عهدنامه‌های بین‌المللی، هم دوجانبه و هم چند جانبه، نقض کردند.
1716 | 61 | وب سایت خبری سرگرمی TMZ می‌داند که عکاس وسیله نقلیه خود را در آن طرف بلوار Sepulveda متوقف کرده و سعی کرده است قبل از عبور از جاده و ادامه کار، از ایست بازرسی پلیس عکس بگیرد، و باعث شد که افسر پلیس گشت بزرگراه کالیفرنیا که در حال انجام ایست بازرسی بود، دو بار به او دستور دهد که برگردد.
1932 | 57 | پنج دقیقه از زمان نمایش بادی شروع به وزش می کند، حدود یک دقیقه بعد، باد به 70 کیلومتر در ساعت می رسد ... سپس باران می آید، اما آنقدر سخت و آنقدر بزرگ است که مانند سوزن به پوست شما سیلی می‌زند، سپس تگرگ از آسمان می‌بارد، مردم وحشت زده ، فریاد می کشند و فرار می کنند.
1735 | 53 | محققان دانشگاه پرینستون آمریكا و دانشگاه اوپسالا در سوئد گونه‌های جدید تکا

In [23]:
METADATA_PATH = (
    DATA_DIR /
    "fleurs_fa_ir_test_metadata.csv"
)

QA_CANDIDATES_PATH = (
    DATA_DIR /
    "fleurs_fa_ir_qa_candidates.csv"
)

df.to_csv(
    METADATA_PATH,
    index=False,
    encoding="utf-8-sig",
)

qa_candidates.to_csv(
    QA_CANDIDATES_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:")
print(METADATA_PATH)
print(QA_CANDIDATES_PATH)

Saved:
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/fleurs_fa_ir_test_metadata.csv
/content/drive/MyDrive/persian_asr_llm_error_propagation/data/fleurs_fa_ir_qa_candidates.csv


In [24]:
import hashlib

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(
                1024 * 1024
            ),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


print(
    "Metadata SHA256:",
    sha256_file(
        METADATA_PATH
    )
)

print(
    "QA candidates SHA256:",
    sha256_file(
        QA_CANDIDATES_PATH
    )
)

Metadata SHA256: bdaa167234bef8f4d9c5b96709dc0f904d62975e38cbd232df73883b7ddc5629
QA candidates SHA256: 9bd17346223f8d3f88b1e0778cc2829126c0293cad559904088bc04b34c729be


In [25]:
word_count.describe()

NameError: name 'word_count' is not defined

In [26]:
qa_generation_input = qa_candidates[
    [
        "id",
        "raw_transcription",
        "transcription",
        "num_recordings",
        "word_count",
        "char_count",
    ]
].copy()

qa_generation_input = (
    qa_generation_input
    .sort_values("id")
    .reset_index(drop=True)
)

qa_generation_input["generation_index"] = (
    range(1, len(qa_generation_input) + 1)
)

print("Candidate semantic items:", len(qa_generation_input))

display(
    qa_generation_input.head()
)

Candidate semantic items: 324


,id,raw_transcription,transcription,num_recordings,word_count,char_count,generation_index
0,1660,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,2,18,99,1
1,1661,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,3,18,86,2
2,1662,آلیاژها اساساً‌ ترکیبی از دو یا چند فلز می‌باش...,آلیاژها اساساً ترکیبی از دو یا چند فلز می‌باشن...,3,20,106,3
3,1663,در Cochamó - مقصد برتر کوهنوردی در شیلی است، ک...,در cochamó - مقصد برتر کوهنوردی در شیلی است، ک...,1,24,128,4
4,1664,دو پودر خشک را مخلوط کنید و سپس با دستان خشک و...,دو پودر خشک را مخلوط کنید و سپس با دستان خشک و...,3,22,88,5


In [29]:
from pathlib import Path

BASE_DIR = Path(
    "/content/drive/MyDrive/persian_asr_llm_error_propagation"
)

DATA_DIR = BASE_DIR / "data"
QA_DIR = BASE_DIR / "qa"
ASR_DIR = BASE_DIR / "asr"
RESULTS_DIR = BASE_DIR / "results"

for p in [DATA_DIR, QA_DIR, ASR_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("QA_DIR:", QA_DIR)

BASE_DIR: /content/drive/MyDrive/persian_asr_llm_error_propagation
QA_DIR: /content/drive/MyDrive/persian_asr_llm_error_propagation/qa


In [30]:
QA_INPUT_PATH = (
    QA_DIR /
    "fleurs_fa_ir_qa_generation_input.jsonl"
)

qa_generation_input.to_json(
    QA_INPUT_PATH,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Saved:")
print(QA_INPUT_PATH)

Saved:
/content/drive/MyDrive/persian_asr_llm_error_propagation/qa/fleurs_fa_ir_qa_generation_input.jsonl


In [31]:
import hashlib

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


qa_input_sha = sha256_file(QA_INPUT_PATH)

print("SHA256:")
print(qa_input_sha)

SHA256:
66e8922a7e8ef5aeeb1feca060e5fd1d60f9d615ba76fb682a7f9331f8b78f01


In [32]:
recording_manifest = df[
    [
        "id",
        "file_name",
        "audio_path",
        "gender",
        "duration_sec",
        "raw_transcription",
        "transcription",
    ]
].copy()

recording_manifest = (
    recording_manifest
    .sort_values(
        ["id", "file_name"]
    )
    .reset_index(drop=True)
)

MANIFEST_PATH = (
    DATA_DIR /
    "fleurs_fa_ir_recording_manifest.csv"
)

recording_manifest.to_csv(
    MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Recordings:", len(recording_manifest))
print("Semantic IDs:", recording_manifest["id"].nunique())
print("Saved:", MANIFEST_PATH)

Recordings: 871
Semantic IDs: 324
Saved: /content/drive/MyDrive/persian_asr_llm_error_propagation/data/fleurs_fa_ir_recording_manifest.csv


In [36]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import hashlib
import unicodedata
import re

BASE_DIR = Path(
    "/content/drive/MyDrive/persian_asr_llm_error_propagation"
)

QA_DIR = BASE_DIR / "qa"
QA_DIR.mkdir(parents=True, exist_ok=True)

GENERATED_QA_PATH = (
    QA_DIR /
    "fleurs_fa_ir_generated_qa_candidates_v1.csv"
)

print("Exists:", GENERATED_QA_PATH.exists())
print(GENERATED_QA_PATH)

Exists: True
/content/drive/MyDrive/persian_asr_llm_error_propagation/qa/fleurs_fa_ir_generated_qa_candidates_v1.csv


In [37]:
qa = pd.read_csv(
    GENERATED_QA_PATH
)

print("Total source texts:", len(qa))

print("\nGeneration status:")
print(
    qa["review_status"]
    .value_counts(dropna=False)
)

print("\nUsable:")
print(
    qa["usable"]
    .value_counts(dropna=False)
)

display(qa.head())

Total source texts: 324

Generation status:
review_status
accept_high_confidence    236
manual_review              50
reject                     38
Name: count, dtype: int64

Usable:
usable
True     286
False     38
Name: count, dtype: int64


,generation_index,id,num_recordings,word_count,char_count,raw_transcription,transcription,usable,question,gold_answer,answer_type,quality_score,review_status,rejection_reason,answer_exact_in_raw,answer_word_count,answer_in_question,qa_generation_model,qa_schema_version
0,1,1660,2,18,99,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,رمانتیسم عنصر عظیمی از جبرگرایی فرهنگی داشت که...,True,جبرگرایی فرهنگی رمانتیسم از نویسندگانی مانند چ...,«گوته»،‌ «فیشته» و «اشلگل»,person_list,5,accept_high_confidence,NaN,True,4,False,GPT-5.6 Sol,1.0
1,2,1661,3,18,86,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,او رقمی برای این کاهش‌ها تعیین نکرد و گفت که ب...,False,NaN,NaN,NaN,2,reject,The utterance contains unresolved references (...,NaN,0,NaN,GPT-5.6 Sol,1.0
2,3,1662,3,20,106,آلیاژها اساساً‌ ترکیبی از دو یا چند فلز می‌باش...,آلیاژها اساساً ترکیبی از دو یا چند فلز می‌باشن...,True,آلیاژها اساساً ترکیبی از چه چیزی هستند؟,دو یا چند فلز,definition,5,accept_high_confidence,NaN,True,4,False,GPT-5.6 Sol,1.0
3,4,1663,1,24,128,در Cochamó - مقصد برتر کوهنوردی در شیلی است، ک...,در cochamó - مقصد برتر کوهنوردی در شیلی است، ک...,True,کوچامو به چه لقبی معروف است؟,یوسمیتی آمریکای جنوبی,entity,5,accept_high_confidence,NaN,True,3,False,GPT-5.6 Sol,1.0
4,5,1664,3,22,88,دو پودر خشک را مخلوط کنید و سپس با دستان خشک و...,دو پودر خشک را مخلوط کنید و سپس با دستان خشک و...,True,دو پودر خشک پس از مخلوط شدن باید به چه شکلی در...,توپی,object,4,manual_review,NaN,True,1,False,GPT-5.6 Sol,1.0


In [38]:
usable = qa[
    qa["usable"] == True
].copy()

print("Usable candidates:", len(usable))

assert len(qa) == 324
assert qa["id"].nunique() == 324
assert len(usable) == 286

assert usable["question"].notna().all()
assert usable["gold_answer"].notna().all()
assert usable["raw_transcription"].notna().all()

print("Basic structure: PASS")

Usable candidates: 286
Basic structure: PASS


In [39]:
usable["exact_span_check"] = usable.apply(
    lambda row:
        str(row["gold_answer"])
        in str(row["raw_transcription"]),
    axis=1,
)

print(
    usable["exact_span_check"]
    .value_counts()
)

assert usable[
    "exact_span_check"
].all()

print("Exact answer span: PASS")

exact_span_check
True    286
Name: count, dtype: int64
Exact answer span: PASS


In [40]:
usable["gold_answer_words"] = (
    usable["gold_answer"]
    .astype(str)
    .str.split()
    .str.len()
)

print(
    usable["gold_answer_words"]
    .describe()
)

print(
    "\nAnswers longer than 12 words:",
    (usable["gold_answer_words"] > 12).sum()
)

count    286.000000
mean       3.597902
std        2.432787
min        1.000000
25%        2.000000
50%        3.000000
75%        5.000000
max       12.000000
Name: gold_answer_words, dtype: float64

Answers longer than 12 words: 0


In [41]:
REVIEW_SEED = 2026

review_df = (
    usable.sample(
        frac=1,
        random_state=REVIEW_SEED
    )
    .reset_index(drop=True)
    .copy()
)

review_df["review_order"] = (
    np.arange(
        1,
        len(review_df) + 1
    )
)

review_df["human_decision"] = "pending"

review_df["human_question"] = (
    review_df["question"]
)

review_df["human_gold_answer"] = (
    review_df["gold_answer"]
)

review_df["human_notes"] = ""

review_df["human_edited"] = False

In [42]:
REVIEW_CHECKPOINT = (
    QA_DIR /
    "qa_human_review_checkpoint.csv"
)

review_df.to_csv(
    REVIEW_CHECKPOINT,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Review items:",
    len(review_df)
)

print(
    "Saved:",
    REVIEW_CHECKPOINT
)

Review items: 286
Saved: /content/drive/MyDrive/persian_asr_llm_error_propagation/qa/qa_human_review_checkpoint.csv


In [43]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import html
import os

In [44]:
def atomic_save_review(df, path):
    tmp = str(path) + ".tmp"

    df.to_csv(
        tmp,
        index=False,
        encoding="utf-8-sig",
    )

    os.replace(
        tmp,
        path
    )

In [45]:
review_df = pd.read_csv(
    REVIEW_CHECKPOINT
)

review_df["human_notes"] = (
    review_df["human_notes"]
    .fillna("")
)

review_df["human_decision"] = (
    review_df["human_decision"]
    .fillna("pending")
)

print(
    "Already reviewed:",
    (
        review_df["human_decision"]
        != "pending"
    ).sum()
)

print(
    "Remaining:",
    (
        review_df["human_decision"]
        == "pending"
    ).sum()
)

Already reviewed: 0
Remaining: 286


In [46]:
output = widgets.Output()

progress = widgets.HTML()

question_box = widgets.Textarea(
    description="Question:",
    layout=widgets.Layout(
        width="100%",
        height="90px"
    )
)

answer_box = widgets.Textarea(
    description="Answer:",
    layout=widgets.Layout(
        width="100%",
        height="70px"
    )
)

notes_box = widgets.Textarea(
    description="Notes:",
    layout=widgets.Layout(
        width="100%",
        height="60px"
    )
)

accept_button = widgets.Button(
    description="Accept",
    button_style="success"
)

accept_edit_button = widgets.Button(
    description="Accept edited",
    button_style="info"
)

reject_button = widgets.Button(
    description="Reject",
    button_style="danger"
)

skip_button = widgets.Button(
    description="Skip"
)

current_index = None


def pending_indices():
    return review_df.index[
        review_df["human_decision"]
        == "pending"
    ].tolist()


def update_progress():
    reviewed = (
        review_df["human_decision"]
        != "pending"
    ).sum()

    accepted = (
        review_df["human_decision"]
        == "accept"
    ).sum()

    rejected = (
        review_df["human_decision"]
        == "reject"
    ).sum()

    progress.value = (
        f"<b>Progress:</b> "
        f"{reviewed}/{len(review_df)}"
        f" &nbsp; | &nbsp; "
        f"Accepted: {accepted}"
        f" &nbsp; | &nbsp; "
        f"Rejected: {rejected}"
    )


def show_item(idx=None):
    global current_index

    pending = pending_indices()

    if idx is None:
        if not pending:
            current_index = None

            with output:
                clear_output()
                print("All QA candidates reviewed.")

            update_progress()
            return

        idx = pending[0]

    current_index = idx

    row = review_df.loc[idx]

    question_box.value = str(
        row["human_question"]
    )

    answer_box.value = str(
        row["human_gold_answer"]
    )

    notes_box.value = str(
        row["human_notes"]
    )

    with output:
        clear_output()

        print("=" * 100)

        print(
            "FLEURS ID:",
            row["id"]
        )

        print(
            "Recordings:",
            row["num_recordings"]
        )

        print()

        print("SOURCE TRANSCRIPT:")
        print()

        print(
            row["raw_transcription"]
        )

        print()

        print("=" * 100)

    update_progress()


def validate_before_accept():
    if current_index is None:
        return False

    row = review_df.loc[
        current_index
    ]

    q = question_box.value.strip()
    a = answer_box.value.strip()

    source = str(
        row["raw_transcription"]
    )

    if not q:
        print("Question cannot be empty.")
        return False

    if not a:
        print("Answer cannot be empty.")
        return False

    if a not in source:
        print(
            "ERROR: edited answer is not "
            "an exact span of the transcript."
        )
        return False

    if a in q:
        print(
            "WARNING: the exact gold answer "
            "appears inside the question."
        )
        return False

    return True


def save_current(decision, edited=False):

    global current_index

    if current_index is None:
        return

    review_df.loc[
        current_index,
        "human_decision"
    ] = decision

    review_df.loc[
        current_index,
        "human_question"
    ] = question_box.value.strip()

    review_df.loc[
        current_index,
        "human_gold_answer"
    ] = answer_box.value.strip()

    review_df.loc[
        current_index,
        "human_notes"
    ] = notes_box.value.strip()

    review_df.loc[
        current_index,
        "human_edited"
    ] = edited

    atomic_save_review(
        review_df,
        REVIEW_CHECKPOINT
    )


def on_accept(_):

    if not validate_before_accept():
        return

    save_current(
        "accept",
        edited=False
    )

    show_item()


def on_accept_edit(_):

    if not validate_before_accept():
        return

    row = review_df.loc[
        current_index
    ]

    changed = (
        question_box.value.strip()
        != str(row["question"]).strip()
        or
        answer_box.value.strip()
        != str(row["gold_answer"]).strip()
    )

    save_current(
        "accept",
        edited=changed
    )

    show_item()


def on_reject(_):

    save_current(
        "reject",
        edited=False
    )

    show_item()


def on_skip(_):

    global current_index

    pending = pending_indices()

    if current_index not in pending:
        show_item()
        return

    pos = pending.index(
        current_index
    )

    if pos + 1 < len(pending):
        show_item(
            pending[pos + 1]
        )


accept_button.on_click(
    on_accept
)

accept_edit_button.on_click(
    on_accept_edit
)

reject_button.on_click(
    on_reject
)

skip_button.on_click(
    on_skip
)

In [47]:
display(
    progress,
    output,
    question_box,
    answer_box,
    notes_box,
    widgets.HBox(
        [
            accept_button,
            accept_edit_button,
            reject_button,
            skip_button,
        ]
    ),
)

show_item()

FLEURS ID: 1709
Recordings: 3

SOURCE TRANSCRIPT:

این گزارش بسیار حیاتی، تقریباً در مورد تمام جنبه‌های سیاست کنونی اجرایی در قبال عراق است و خواستار تغییر جهت فوری است.



In [51]:
review_df = pd.read_csv(
    REVIEW_CHECKPOINT
)

print(
    review_df[
        "human_decision"
    ].value_counts(
        dropna=False
    )
)

human_decision
pending    240
accept      46
Name: count, dtype: int64


In [49]:
remaining = review_df[
    review_df["human_decision"]
    == "pending"
]

print(
    "Remaining:",
    len(remaining)
)

Remaining: 242


In [50]:
review_df = pd.read_csv(
    REVIEW_CHECKPOINT
)

assert (
    review_df["human_decision"]
    != "pending"
).all()

print(
    review_df[
        "human_decision"
    ].value_counts()
)

AssertionError: 

In [ ]:
final_qa = review_df[
    review_df["human_decision"]
    == "accept"
].copy()

final_qa["question"] = (
    final_qa["human_question"]
)

final_qa["gold_answer"] = (
    final_qa["human_gold_answer"]
)

final_qa["qa_id"] = (
    "fleurs_fa_ir_"
    + final_qa["id"]
      .astype(str)
)

print(
    "Final QA examples:",
    len(final_qa)
)

In [ ]:
assert final_qa["id"].is_unique
assert final_qa["qa_id"].is_unique
assert final_qa["question"].notna().all()
assert final_qa["gold_answer"].notna().all()

final_qa["final_exact_span"] = (
    final_qa.apply(
        lambda row:
            str(row["gold_answer"])
            in str(
                row["raw_transcription"]
            ),
        axis=1,
    )
)

assert final_qa[
    "final_exact_span"
].all()

print("Final exact-span check: PASS")

In [ ]:
duplicate_questions = (
    final_qa[
        final_qa["question"]
        .duplicated(keep=False)
    ]
)

print(
    "Duplicate questions:",
    len(duplicate_questions)
)

if len(duplicate_questions):
    display(
        duplicate_questions[
            [
                "id",
                "question",
                "gold_answer",
            ]
        ]
    )

In [ ]:
print(
    final_qa[
        "answer_type"
    ]
    .value_counts()
)

In [ ]:
FINAL_COLUMNS = [
    "qa_id",
    "id",
    "num_recordings",
    "raw_transcription",
    "transcription",
    "question",
    "gold_answer",
    "answer_type",
    "human_edited",
]

In [ ]:
frozen_qa = final_qa[
    FINAL_COLUMNS
].copy()

In [ ]:
FINAL_JSONL = (
    QA_DIR /
    "qa_benchmark_v1.jsonl"
)

FINAL_CSV = (
    QA_DIR /
    "qa_benchmark_v1.csv"
)

frozen_qa.to_json(
    FINAL_JSONL,
    orient="records",
    lines=True,
    force_ascii=False,
)

frozen_qa.to_csv(
    FINAL_CSV,
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
AUDIT_PATH = (
    QA_DIR /
    "qa_human_review_audit_v1.csv"
)

review_df.to_csv(
    AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda:
                f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


print(
    "QA benchmark:",
    FINAL_JSONL
)

print(
    "Examples:",
    len(frozen_qa)
)

print(
    "SHA256:",
    sha256_file(
        FINAL_JSONL
    )
)